# TableTrace GPU 실험: EXAONE LoRA → vLLM / SGLang → 멀티 GPU

**이 노트북은 Colab 또는 Kaggle에서 실행합니다. 작성자가 실행해 보지 못했습니다**(개발 컨테이너에 GPU가 없고 Hugging Face가 막혀 있음). 아래 명령은 저장소의 스크립트를 부르는 것이고, 스크립트의 학습 루프·엔진 연결은 CPU에서 작은 모델과 모의 서버로 시험했습니다(`tests/test_lora_sft_smoke.py`, `tests/test_engines.py`). 실제 EXAONE·vLLM·SGLang·NCCL 실행에서 생기는 문제는 여기서 처음 드러날 수 있습니다.

| 하고 싶은 것 | 필요한 것 | Colab 무료(T4 1장) | Kaggle(T4 2장) |
|---|---|---|---|
| 1. EXAONE LoRA 튜닝 | GPU 1장 | 가능 | 가능 |
| 2. vLLM 서빙·평가·벤치 | GPU 1장 | 가능 | 가능 |
| 3. SGLang 서빙·평가·벤치 | **Ampere 이상 GPU 1장**(L4·A100·H100) | **T4에서는 실패 확인**(아래 3번 절) | **불가**(T4·P100만 제공) |
| 4. 텐서 병렬 TP=2 | **GPU 2장** | 불가 | 가능 |
| 5. 멀티 GPU 모델 병렬(mT5 2단계 파이프라인) | **GPU 2장** | 불가 | 가능 |

**남은 실험 두 세션**: (가) Kaggle T4 ×2에서 `LIMIT = 0`으로 1~5번 절 전부 → 전체 데이터 LoRA, vLLM(1장·TP=2), SGLang(우회), 수작업 30문항, 파이프라인. (나) Colab 유료 L4 또는 A100에서 `LIMIT = 0`으로 0~3번 절 + 5번 절 → SGLang 대 vLLM 제 성능 비교(4번 절은 GPU 1장이라 건너뜀). 두 세션의 결과 파일을 모두 보내 주세요(`experiments/run_info.json`이 어느 환경인지 기록).

T4는 bfloat16을 지원하지 않습니다(vLLM 0.32는 T4에서 float16·float32만 목록에 올림). 그래서 아래는 fp16입니다. 라이선스: EXAONE은 비상업 연구용(EXAONE AI Model License 1.2-NC)이고 Hugging Face에서 동의가 필요할 수 있습니다. 토큰은 Colab **Secrets**(`HF_TOKEN`)에 넣으세요.

### Kaggle에서 하는 경우 (GPU 2장이 필요한 4번 절까지 하려면 Kaggle)

1. kaggle.com 계정에서 **휴대폰 인증**을 마쳐야 GPU와 인터넷을 켤 수 있습니다.
2. Notebooks → *New Notebook* → File → *Import Notebook*에서 이 `.ipynb`를 올리거나 GitHub 주소를 붙입니다.
3. 오른쪽 **Session options**: Accelerator = *GPU T4 x2*, Internet = *On*.
4. Add-ons → **Secrets**에 `HF_TOKEN`을 추가하고 이 노트북에 연결합니다.
5. 위에서 아래로 실행합니다. 무료 GPU는 주 30시간, 세션은 최대 12시간입니다. 결과는 마지막 셀이 `/kaggle/working`에 만드는 `gpu_results.tgz`입니다(Output 탭에서 받기).
6. 디스크: `/kaggle/working`은 약 20GB로 제한되어 있어 vLLM·SGLang 가상환경은 `/tmp`에 만듭니다(아래 셀). 그래도 모자라면 vLLM을 끝낸 뒤 `rm -rf /tmp/venv-vllm`으로 지우고 SGLang을 설치하세요.
7. 1~3번 절은 GPU 한 장만 씁니다. 4번 절에서 두 장을 씁니다.

In [ ]:
!nvidia-smi
import torch
print("GPU 수:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("compute capability:", torch.cuda.get_device_capability(0), "(7.5 = T4: bf16 없음 → fp16)")

## 0. 준비 — 저장소, 의존성, 토큰

In [ ]:
import os, subprocess
try:                                   # Colab Secrets
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    try:                               # Kaggle Secrets
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("HF_TOKEN을 Secrets에 넣지 않았습니다. 공개 모델만 받을 수 있습니다.")

REPO = "https://github.com/sokldjs554/tabletrace"      # 비공개 저장소면 https://<토큰>@github.com/... 로 바꾸세요
ROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"      # Kaggle / Colab. Kaggle 이미지에도 /content 폴더가 있어서 /kaggle을 먼저 본다(Output 패널은 /kaggle/working만 보여 줌)
VENV = ROOT if ROOT == "/content" else "/tmp"                                  # Kaggle: working dir is only ~20GB
if not os.path.isdir(f"{ROOT}/tabletrace"):
    subprocess.run(["git", "clone", "-q", REPO, f"{ROOT}/tabletrace"], check=True)
os.chdir(f"{ROOT}/tabletrace")
!pip -q install -e ".[dev]" peft accelerate sentencepiece
!pip -q uninstall -y torchao      # Colab의 torchao 0.10.0은 peft가 요구하는 0.16 이상보다 낮아 LoRA 주입에서 ImportError. 이 프로젝트는 torchao를 쓰지 않음

## 1. EXAONE LoRA 튜닝 (같은 데이터, 같은 프로그램 문법)

- `LGAI-EXAONE/EXAONE-4.0-1.2B`는 transformers가 기본 지원하고 T4 메모리에 여유가 있습니다. `EXAONE-3.5-2.4B-Instruct`를 쓰려면 `TRUST="--trust-remote-code"`를 켜고, transformers 버전 충돌이 나면 4.0-1.2B로 돌아가세요.
- 학습 시간 **실측**: Kaggle T4 1장에서 전체 데이터(3,587개, 224 step, fp16) 1 epoch이 **31분**이었습니다(`experiments/exaone_lora_full/train_info.json`). 먼저 `LIMIT=300`으로 몇 분짜리 시험을 하고 싶으면 값을 바꾸세요(결과 파일 이름이 `exaone_*`로 달라집니다).
- 본문 예제(`text2_*`)는 KorQuAD 파생이라 저장소에 없습니다. 여기서는 표 조회·계산·보류 예제만 씁니다.

In [ ]:
import torch
MODEL = "LGAI-EXAONE/EXAONE-4.0-1.2B"
TRUST = ""            # EXAONE-3.5 라면 "--trust-remote-code"
LIMIT = 0             # 시험: 300, 본 학습: 0 (전부 3,654개). 결과 파일 이름은 LIMIT=0이면 exaone_full_*, 아니면 exaone_*
GPU_NAME = torch.cuda.get_device_name(0).replace(" ", "_")       # 평가 결과의 meta.label에 남음 (T4 / L4 / A100 구분)
GPU_TAG = "" if "T4" in GPU_NAME else ("_l4" if "L4" in GPU_NAME else "_a100" if "A100" in GPU_NAME else "_gpu")   # T4(Kaggle) 결과와 파일 이름이 겹치지 않게
TAG = ("" if LIMIT else "_full") + GPU_TAG
LORA_DIR = f"experiments/exaone_lora{TAG}"
print("GPU:", GPU_NAME, "| 결과 파일 접미사:", TAG or "(없음)")
!CUDA_VISIBLE_DEVICES=0 python scripts/train/exaone_lora_sft.py --model {MODEL} {TRUST} --dtype auto --grad-ckpt --merge \
    --train data/ko/train.jsonl data/ko/synthetic_train.jsonl --corpus data/ko/documents.jsonl \
    --epochs 1 --bs 2 --grad-accum 8 --limit {LIMIT} --out {LORA_DIR}
!python -c "import json; t=json.load(open('{LORA_DIR}/train_info.json')); print('학습 시간', round(t.get('wall_s', 0)/60, 1), '분,', t['n_examples'], '예제,', t['steps'], 'step,', t['dtype'])"


## 2. vLLM — 서빙, 같은 검증기로 평가, 벤치

vLLM과 SGLang은 torch 버전을 다르게 고정하므로 **가상환경을 따로** 만듭니다(Colab 기본 torch를 건드리지 않음). 설치에 몇 분 걸립니다.

In [ ]:
!pip -q install uv
!uv venv --clear {VENV}/venv-vllm -q
!uv pip install -q --python {VENV}/venv-vllm/bin/python vllm

In [ ]:
import os, subprocess, time, requests

REPO_DIR = f"{ROOT}/tabletrace"
os.chdir(REPO_DIR)      # 서버와 평가 명령이 항상 저장소 폴더에서 시작하도록 고정 (cwd가 다르면 상대 경로 모델 폴더를 못 찾는다)

def stop_servers():
    """이전에 띄운 vLLM/SGLang 서버가 남아 있으면 GPU 메모리와 포트를 쥐고 있어 다음 서버가 뜨지 않는다. 먼저 모두 끈다."""
    pats = ("vllm serve", "vllm.entrypoints", "VLLM::", "sglang.launch_server", "sglang::")
    for sig in ("-TERM", "-KILL"):
        for pat in pats:
            subprocess.run(["pkill", sig, "-f", pat], capture_output=True)    # 인자 목록으로 호출: 셸 명령줄에 패턴이 남지 않아 자기 자신을 죽이지 않음
        time.sleep(6)
    print(subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

def serve(script, venv, port, log, **env):
    stop_servers()
    e = dict(os.environ, PORT=str(port), PATH=f"{venv}/bin:" + os.environ["PATH"], PYTHONPATH="", PYTHONNOUSERSITE="1", **{k: str(v) for k, v in env.items()})   # Colab의 PYTHONPATH/sitecustomize가 가상환경에 새지 않게
    proc = subprocess.Popen(["bash", script], env=e, stdout=open(log, "w"), stderr=subprocess.STDOUT, cwd=REPO_DIR)
    for _ in range(240):
        try:
            requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=2).raise_for_status()
            return proc
        except Exception:
            if proc.poll() is not None:
                break
            time.sleep(5)
    tail = "".join(open(log).readlines()[-80:])
    raise RuntimeError("서버가 뜨지 않았습니다(메모리 부족이면 MAX_LEN을 줄이거나 EXTRA_ARGS='--enforce-eager'). 서버 로그 마지막 80줄:\n" + tail)

MERGED = f"{REPO_DIR}/{LORA_DIR}/merged"    # 절대 경로(상대 경로면 서버 프로세스의 cwd에 따라 OSError). 1번 절(LoRA 튜닝)의 결과. 튜닝 없이 서버만 시험하려면 "LGAI-EXAONE/EXAONE-4.0-1.2B"로 바꾸세요(미튜닝 모델이라 점수는 낮음)
if "/" in MERGED and not MERGED.startswith("LGAI-") and not os.path.isfile(f"{MERGED}/config.json"):
    raise FileNotFoundError(f"{MERGED} 가 없습니다. 1번 절(LoRA 튜닝)이 끝나지 않았거나 실패했습니다. `!ls -la {REPO_DIR}/{LORA_DIR}` 로 확인하고 1번 절 출력의 마지막 줄을 보세요.")
vllm_proc = serve("scripts/serving/vllm_tp2.sh", f"{VENV}/venv-vllm", 8001, f"{ROOT}/vllm.log",
                  MODEL=MERGED, NUM_GPUS=1, DTYPE="half", MAX_LEN=2048)

In [ ]:
# 학습한 EXAONE 플래너를 같은 검증기·같은 지표로 평가 (mT5와 비교 가능한 표가 나옴). ko test 699문항 + 수작업 30문항.
# 재검증(--replay-from)에도 같은 --tokenizer와 --max-in을 줘야 행 예산이 본 실행과 같아진다(기본값 512는 다른 값을 낸다).
def evaluate(url, name, concurrency):
    for data, suffix in (("data/ko/test.jsonl data/ko/synthetic_test.jsonl", "ko_test"), ("data/ko/custom_eval.jsonl", "ko_custom")):
        out = f"experiments/eval/exaone{TAG}_{name}_{suffix}"
        !python -m tabletrace.eval.run --planner engine --engine-url {url} --engine-model planner --label {GPU_NAME} \
            --tokenizer {MERGED} --max-in 672 --concurrency {concurrency} --corpus data/ko/documents.jsonl --data {data} --out {out} --no-entity-check
        !python -m tabletrace.eval.run --planner engine --replay-from {out} --label {GPU_NAME} --tokenizer {MERGED} --max-in 672 --corpus data/ko/documents.jsonl --data {data} --out {out}_ec
    for c in (1, 8, 32):
        !python -m tabletrace.bench.engine_bench --url {url} --model planner --n 64 --concurrency {c} --tokenizer {MERGED} --out experiments/bench/engine{TAG}_{name}_c{c}.json

evaluate("http://127.0.0.1:8001", "vllm", 16)
vllm_proc.terminate(); vllm_proc.wait()


## 3. SGLang — T4에서는 우회 모듈로, Ampere 이상에서는 그대로

T4(sm75)에서 `sglang==0.5.9`는 모델을 올린 뒤 첫 연산(`sgl_kernel.rmsnorm`)에서 `no kernel image is available for execution on the device`로 멈춥니다(2026-11-04·11-07 두 번 확인, `experiments/serving/sglang_t4_failure.md`). 미리 컴파일된 `sgl_kernel`에 T4용 코드가 없기 때문입니다. 그래서 아래 셀은 T4를 만나면 `scripts/serving/sglang_t4_shim.py`를 가상환경에 설치하고 `SGLANG_T4_SHIM=1`로 서버를 띄웁니다. 이 모듈은 SGLang의 연산 클래스(`MultiPlatformOp`)가 fused CUDA 커널 대신 자체 PyTorch 구현(`forward_native`)을 쓰게 하고, 어텐션은 Triton 백엔드, 샘플링은 PyTorch 백엔드로 둡니다. SGLang 자체 커널보다 느리고 **작성자는 GPU가 없어 실행해 보지 못했습니다.** 실패하면 아래 진단 셀 출력을 알려 주세요. Ampere 이상(L4·A100·H100)에서는 우회 없이 그대로 돕니다. **vLLM과 SGLang의 제 성능 비교는 Colab의 L4/A100에서 0~3번 절(+5번 절)을 돌려야 합니다**(T4의 SGLang 숫자는 우회 모듈 탓에 비교가 아님). Ampere 이상에서는 두 엔진 모두 기본 커널(vLLM: FlashAttention/FlashInfer 자동, SGLang: flashinfer)로 돌고, 결과 파일은 `exaone_full_l4_*`처럼 GPU 접미사가 붙어 T4 결과와 섞이지 않습니다. 비교 조건: 같은 병합 모델, 같은 GPU, fp16, 같은 64문항·동시 1/8/32, OpenAI 호환 API 왕복 지연.

설치 메모: Colab·Kaggle 기본 파이썬은 3.13이라 `sglang[all]`이 소스 빌드로 넘어가 실패합니다. 파이썬 3.12 가상환경에 `sglang==0.5.9`로 고정해 설치하면 wheel만으로 설치됩니다. 최신 0.5.23은 pre-release 의존성(`flash-attn-4`) 때문에 기본 설정으로는 해석되지 않습니다.


In [ ]:
# SGLang은 transformers 4.57을 쓰는데, 학습 쪽 transformers 5.x가 저장한 tokenizer_config.json(AttributeError: 'list' object has no attribute 'keys')과
# config.json(RoPE 설정이 rope_parameters에만 있어 4.57은 rope_theta 10,000·스케일링 없음으로 읽음 → 문맥을 무시하는 출력)을 4.57도 읽게 고친다.
# 양쪽이 읽는 형식으로 바꾸며, 이미 바뀌어 있으면 아무 일도 하지 않는다.
!git pull -q
!python scripts/serving/fix_tokenizer_config.py experiments/exaone_lora experiments/exaone_lora/merged

In [ ]:
import os, torch
!git pull -q 2>/dev/null || echo "git pull 생략(로컬 결과 파일과 충돌). 아래는 저장소에 이미 있는 파일을 씁니다."
T4 = torch.cuda.get_device_capability(0)[0] < 8          # Turing(T4) 이하면 우회 모듈 사용
print("GPU:", torch.cuda.get_device_name(0), "compute capability", torch.cuda.get_device_capability(0), "→ 우회 모듈", "사용" if T4 else "불필요")
if not os.path.isfile(f"{VENV}/venv-sglang/bin/python"):
    !uv venv --clear --python 3.12 {VENV}/venv-sglang -q
    !uv pip install --python {VENV}/venv-sglang/bin/python "sglang==0.5.9" 2>&1 | tail -n 3
if T4:
    !PYTHONPATH= {VENV}/venv-sglang/bin/python scripts/serving/sglang_t4_shim.py --install
    !PYTHONPATH= SGLANG_T4_SHIM=1 {VENV}/venv-sglang/bin/python scripts/serving/sglang_t4_shim.py --check
!python scripts/serving/fix_tokenizer_config.py {MERGED}      # config.json의 RoPE 키 복원(안 하면 SGLang이 문맥을 무시하는 답을 냄, 이슈 39)
SG_EXTRA = ""        # CUDA 그래프 캡처에서 실패하면 "--disable-cuda-graph" 로 바꿔 다시 실행 (느려지지만 한 단계를 건너뜀)
# T4: 우회 모듈 + triton 어텐션 + pytorch 샘플링(sgl_kernel을 피함). Ampere 이상: SGLang 기본 백엔드 그대로(flashinfer) → vLLM과 공정 비교
sg_proc = serve("scripts/serving/sglang.sh", f"{VENV}/venv-sglang", 8002, f"{ROOT}/sglang.log",
                MODEL=MERGED, NUM_GPUS=1, DTYPE="half", ATTN_BACKEND="triton" if T4 else "", SAMPLING_BACKEND="pytorch" if T4 else "flashinfer",
                SGLANG_T4_SHIM="1" if T4 else "0", EXTRA_ARGS=SG_EXTRA)


In [ ]:
# SGLang 서버가 뜨지 않으면 이 셀의 출력 전체를 복사해서 알려 주세요
!nvidia-smi | head -n 4
!PYTHONPATH= {VENV}/venv-sglang/bin/python -c "import sys, torch; print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.version.cuda, 'gpu available', torch.cuda.is_available())"
!uv pip list --python {VENV}/venv-sglang/bin/python 2>/dev/null | grep -i -E "^(sglang|sgl-kernel|flashinfer|torch |triton|transformers)"
!ls {VENV}/venv-sglang/lib/python3.12/site-packages/ | grep -i sglang_t4_shim
!grep -n "sglang_t4_shim\|Error\|error\|Traceback" {ROOT}/sglang.log | head -n 30
!tail -n 60 {ROOT}/sglang.log


In [ ]:
evaluate("http://127.0.0.1:8002", "sglang", 8)
sg_proc.terminate(); sg_proc.wait()


## 4. 멀티 GPU — **Kaggle(T4 ×2) 또는 GPU 2장 이상에서만**

Colab 한 장으로는 아래 두 가지를 할 수 없습니다. Kaggle 노트북 설정에서 Accelerator를 *GPU T4 x2*로, Internet을 켜세요.

(a) **vLLM 텐서 병렬**: 바로 아래 셀입니다. 같은 모델을 GPU 두 장에 나눠 올리고 평가·벤치를 다시 합니다.

(b) **모델 병렬(2단계 파이프라인)**: mT5를 인코더(GPU0)와 디코더+lm_head(GPU1)로 나눠 NCCL로 주고받습니다. 튜닝한 가중치는 git에 없어서 여기서는 **원본 `google/mt5-small`**로 돌립니다. 이 실험이 보는 것은 분할·전송·메모리·"1장과 2장의 출력이 같은가"이지 답변 품질이 아닙니다.

In [ ]:
# (a) vLLM 텐서 병렬 TP=2 — 2번 절(서버를 띄우는 serve 함수와 vLLM 가상환경)을 먼저 실행했어야 합니다.
import torch
if torch.cuda.device_count() < 2:
    raise SystemExit("GPU가 1장이라 4번 절(TP=2, 2단계 파이프라인)은 건너뜁니다. Kaggle T4 x2에서만 실행하세요. 5번 절로 가세요.")
EXTRA = ""     # 서버가 멈추거나 NCCL 오류가 나면 "--enforce-eager --disable-custom-all-reduce" 로 바꿔 다시 실행
tp2_proc = serve("scripts/serving/vllm_tp2.sh", f"{VENV}/venv-vllm", 8003, f"{ROOT}/vllm_tp2.log",
                 MODEL=MERGED, NUM_GPUS=2, DTYPE="half", MAX_LEN=2048, EXTRA_ARGS=EXTRA)
!nvidia-smi --query-gpu=index,memory.used --format=csv
evaluate("http://127.0.0.1:8003", "vllm_tp2", 16)
tp2_proc.terminate(); tp2_proc.wait()


In [ ]:
import torch
if torch.cuda.device_count() < 2:
    raise SystemExit("GPU가 1장이라 2단계 파이프라인(torchrun 2 프로세스)은 건너뜁니다. 5번 절로 가세요.")
!git pull -q
!python -m tabletrace.bench.pipeline_parallel --model google/mt5-small --single --backend nccl --n 64 --bs 8 --out experiments/bench/pp_single_gpu.json
# 두 번째 명령이 실패하면 각 rank의 오류가 위쪽에 출력됩니다. 통째로 보려면 아래 줄 끝에 "2>&1 | tail -n 80"을 붙이세요.
# GPU 간 통신(NCCL)이 멈추거나 P2P 오류가 나면 앞에 NCCL_P2P_DISABLE=1 을 붙여 다시 시도하세요.
!torchrun --nproc_per_node=2 -m tabletrace.bench.pipeline_parallel --model google/mt5-small --backend nccl --n 64 --bs 8 --out experiments/bench/pp_2stage_gpu.json

## 5. 결과 돌려주기

`experiments/eval/exaone_*`, `experiments/bench/engine_*`, `experiments/bench/pp_*_gpu.json`, `experiments/exaone_lora/train_info.json`이 산출물입니다. 압축해서 받거나(아래 셀) 저장소의 새 브랜치에 푸시한 뒤 알려 주세요. 그러면 README·보고서의 "구현 후 미실행"을 실측 값으로 바꾸고 `check_numbers`로 검증합니다.

In [ ]:
import glob, os, subprocess, json, time, torch
json.dump({"platform": "kaggle" if os.path.isdir("/kaggle/working") else "colab", "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
           "limit": LIMIT, "tag": TAG, "finished": time.strftime("%Y-%m-%dT%H:%M:%S")}, open("experiments/run_info.json", "w"), indent=1)
paths = [p for pat in ("experiments/eval/exaone_*", "experiments/bench/engine_*", "experiments/bench/pp_*_gpu.json", "experiments/exaone_lora*/train_info.json", "experiments/run_info.json") for p in sorted(glob.glob(pat))]
print("담는 것:", paths)
out = "/kaggle/working/gpu_results.tgz" if os.path.isdir("/kaggle/working") else "/content/gpu_results.tgz"   # Kaggle은 Output 패널, Colab은 왼쪽 파일 탭에서 받기
r = subprocess.run(["tar", "czf", out, *paths], capture_output=True, text=True)
print(r.stderr or "tar ok", "|", out, os.path.getsize(out), "bytes")
try:
    from google.colab import files; files.download(out)      # 브라우저가 막으면 아래 안내대로 직접 받으세요
except Exception as e:
    print("자동 내려받기 실패:", e)
    print(f"왼쪽 파일 탭(폴더 아이콘)에서 {out} 을 찾아 ⋮ → 다운로드 (Kaggle은 Output 탭)")
